In [ ]:
import torch
import torch.nn as nn

In [ ]:
def DoubleConv(in_chan,out_chan):
    return nn.Sequential(
        nn.Conv2d(in_chan,out_chan,kernel_size=3,padding=1,bias=False),
        nn.BatchNorm2d(out_chan),
        nn.ReLU(inplace=True),

        nn.Conv2d(out_chan,out_chan,kernel_size=3,padding=1,bias=False),
        nn.BatchNorm2d(out_chan),
        nn.ReLU(inplace=True),
    )

In [ ]:
class Unet(nn.Module):
    def __init__(self,n_chan,n_classes):
        super(Unet,self).__init__()

        self.enc1  = DoubleConv(n_chan,64)
        self.pool1 = nn.MaxPool2d(kernel_size=2,stride=2)

        self.enc2  = DoubleConv(64,128)
        self.pool2 = nn.MaxPool2d(kernel_size=2,stride=2)

        self.enc3  = DoubleConv(128,256)
        self.pool3 = nn.MaxPool2d(kernel_size=2,stride=2)

        self.enc4  = DoubleConv(256,512)
        self.pool4 = nn.MaxPool2d(kernel_size=2,stride=2)

        self.bottleneck = DoubleConv(512,1024)

        # Interpolacion
        # self.up = nn.Upsample(scale_factor=2, mode=mode,
        #                       align_corners=False if mode=="bilinear" else None)


        # Conv para preparar canales: out_ch * r^2
        # self.conv_expand = nn.Conv2d(in_ch, out_ch * r * r, kernel_size=3, padding=1)
        # self.shuffle = nn.PixelShuffle(r)

        self.up4  = nn.ConvTranspose2d(1024,512,kernel_size=2,stride=2)
        self.dec4 = DoubleConv(1024,512)

        self.up3  = nn.ConvTranspose2d(512,256,kernel_size=2,stride=2)
        self.dec3 = DoubleConv(512,256)

        self.up2  = nn.ConvTranspose2d(256,128,kernel_size=2,stride=2)
        self.dec2 = DoubleConv(256,128)

        self.up1  = nn.ConvTranspose2d(128,64,kernel_size=2,stride=2)
        self.dec1 = DoubleConv(128,64)

        self.OutLayer = nn.Conv2d(64,n_classes,kernel_size=1)

    def forward(self,x):
        # Encoder
        z1 = self.enc1(x)
        z2 = self.pool1(z1)

        z2 = self.enc2(z2)
        z3 = self.pool2(z2)

        z3 = self.enc3(z3)
        z4 = self.pool3(z3)

        z4 = self.enc4(z4)
        Z  = self.pool4(z4)

        #           0      1 2 3
        # Z: [n_batch,n_chan,h,w]
        Z = self.bottleneck(Z)

        # UP
        y = self.up4(Z)
        y = torch.cat([y, z4], dim=1)
        y = self.dec4(y)

        y = self.up3(y)
        y = torch.cat([y, z3], dim=1)
        y = self.dec3(y)

        y = self.up2(y)
        y = torch.cat([y, z2], dim=1)
        y = self.dec2(y)

        y = self.up1(y)
        y = torch.cat([y, z1], dim=1)
        y = self.dec1(y)

        return self.OutLayer(y)

In [ ]:
model = Unet(3,10)

In [ ]:
print(model)

Unet(
  (enc1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): SiLU(inplace=True)
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): SiLU(inplace=True)
  )
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False

# EfficientUNet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_chan, out_chan):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_chan, out_chan, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_chan),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_chan, out_chan, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_chan),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv_reduce = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        self.double_conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        # Upsample al tamaño espacial del skip
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv_reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.double_conv(x)

In [ ]:
class EfficientUNetB0(nn.Module):
    def __init__(self, n_classes, pretrained=True):
        super().__init__()

        # Backbone EfficientNet-B0 (entrada siempre 3 canales)
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = efficientnet_b0(weights=weights)

        # Desempaquetamos explícitamente los bloques de features
        self.stem   = backbone.features[0]  # 3  -> 32, H/2
        self.block1 = backbone.features[1]  # 32 -> 16, H/2
        self.block2 = backbone.features[2]  # 16 -> 24, H/4
        self.block3 = backbone.features[3]  # 24 -> 40, H/8
        self.block4 = backbone.features[4]  # 40 -> 80, H/16
        self.block5 = backbone.features[5]  # 80 -> 112, H/16
        self.block6 = backbone.features[6]  # 112 -> 192, H/32
        self.block7 = backbone.features[7]  # 192 -> 320, H/32
        self.block8 = backbone.features[8]  # 320 -> 1280, H/32  (conv head)

        # ---- Decoder explícito (sin for) ----
        # Skips (canales conocidos):
        # skip4: x7  (320)  @ H/32
        # skip3: x5  (112)  @ H/16
        # skip2: x3  (40)   @ H/8
        # skip1: x2  (24)   @ H/4
        # skip0: x0  (32)   @ H/2

        # Bottleneck: x8 (1280) @ H/32

        self.up4 = UpBlock(in_ch=1280, skip_ch=320, out_ch=320)  # H/32
        self.up3 = UpBlock(in_ch=320,  skip_ch=112, out_ch=112)  # H/16
        self.up2 = UpBlock(in_ch=112,  skip_ch=40,  out_ch=40)   # H/8
        self.up1 = UpBlock(in_ch=40,   skip_ch=24,  out_ch=24)   # H/4
        self.up0 = UpBlock(in_ch=24,   skip_ch=32,  out_ch=32)   # H/2

        # Capa final: upsample a resolución original + conv 1x1
        self.out_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder EfficientNet-B0
        x0 = self.stem(x)        # [B, 32, H/2,  W/2]
        x1 = self.block1(x0)     # [B, 16, H/2,  W/2]
        x2 = self.block2(x1)     # [B, 24, H/4,  W/4]
        x3 = self.block3(x2)     # [B, 40, H/8,  W/8]
        x4 = self.block4(x3)     # [B, 80, H/16, W/16]
        x5 = self.block5(x4)     # [B, 112,H/16, W/16]
        x6 = self.block6(x5)     # [B, 192,H/32, W/32]
        x7 = self.block7(x6)     # [B, 320,H/32, W/32]
        x8 = self.block8(x7)     # [B, 1280,H/32,W/32]  bottleneck

        # Skips
        skip0 = x0   # 32  @ H/2
        skip1 = x2   # 24  @ H/4
        skip2 = x3   # 40  @ H/8
        skip3 = x5   # 112 @ H/16
        skip4 = x7   # 320 @ H/32

        # Decoder
        y = self.up4(x8, skip4)  # -> 320 @ H/32
        y = self.up3(y,  skip3)  # -> 112 @ H/16
        y = self.up2(y,  skip2)  # -> 40  @ H/8
        y = self.up1(y,  skip1)  # -> 24  @ H/4
        y = self.up0(y,  skip0)  # -> 32  @ H/2

        # Upsample fina
        y = F.interpolate(y, scale_factor=2, mode="bilinear", align_corners=False)
        out = self.out_conv(y)   # [B, n_classes, H, W]

        return out

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [ ]:
model = EfficientUNetB0(n_classes=2, pretrained=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Congelar parte del encoder
for module in [model.stem, model.block1, model.block2, model.block3, model.block4]:
    for p in module.parameters():
        p.requires_grad = False

# Definir parámetros del encoder que sí se van a entrenar
encoder_ft_params = []
for module in [model.block5, model.block6, model.block7, model.block8]:
    encoder_ft_params += list(module.parameters())

# 3) Definir parámetros del decoder (todos se entrenan)
decoder_params = list(model.up4.parameters()) \
               + list(model.up3.parameters()) \
               + list(model.up2.parameters()) \
               + list(model.up1.parameters()) \
               + list(model.up0.parameters()) \
               + list(model.out_conv.parameters())

# Optimizer con distintos learning rates
optimizer = torch.optim.AdamW(
    [
        {"params": encoder_ft_params, "lr": 1e-4},  # encoder: LR pequeño
        {"params": decoder_params,    "lr": 1e-3},  # decoder: LR más grande
    ],
    weight_decay=1e-4,
)

# Ejemplo de pérdida: segmentación binaria / multiclass
criterion = nn.CrossEntropyLoss()  # o BCEWithLogitsLoss si usas 1 canal


In [ ]:
for epoch in range(10):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:   # define tu DataLoader aparte
        images = images.to(device)       # [B, 3, H, W]
        masks  = masks.to(device)        # [B, H, W] (para CrossEntropy)

        optimizer.zero_grad()

        logits = model(images)           # [B, n_classes, H, W]
        loss   = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}: loss = {epoch_loss:.4f}")
